# CS 589 — Homework 2 (ElasticSearch)
This notebook implements the full pipeline from the homework handout:
- Create indices in ElasticSearch for **TF‑IDF, BM25, Dirichlet LM**
- Bulk index StackOverflow questions
- Evaluate using ElasticSearch **Rank Eval** and report **NDCG@10** for 9 settings (3 langs × 3 models)

⚠️ Prereqs:
1) ElasticSearch + Kibana running locally (ES 7.9.x)
2) Dataset files on disk: `python_qid2all.txt`, `python_cosidf.txt`, etc.

Tip: test on a small subset first, then run full indexing/eval.

## 0) Install Python dependency (one-time)
Run this once in your environment (or install via conda/poetry):

In [ ]:
!pip -q install elasticsearch==7.9.1

## 1) Configuration
Set `DATA_DIR` to the folder containing your dataset TXT files.

Expected per language:
- `$lang_qid2all.txt`: all questions (qid + title + body + answer)
- `$lang_cosidf.txt`: relevance judgments (qid1 paired with qid2 + label)


In [ ]:
from pathlib import Path
from typing import Dict, List, Tuple, Iterable, Optional
import json
import re

# ---- ElasticSearch endpoint ----
ES_HOST = "http://localhost:9200"  # change if needed

# ---- Set this to your dataset folder ----
DATA_DIR = Path("PUT_YOUR_DATASET_FOLDER_HERE")  # e.g., Path("/Users/archi/Downloads/cs589_hw2_data")

LANGS = ["python", "java", "javascript"]

QID2ALL = {lang: DATA_DIR / f"{lang}_qid2all.txt" for lang in LANGS}
COSIDF  = {lang: DATA_DIR / f"{lang}_cosidf.txt"  for lang in LANGS}

def index_name(lang: str, model: str) -> str:
    return f"{lang}_{model}"

# TF-IDF: easiest is built-in "classic" similarity.
# If your instructor strictly requires scripted similarity, flip TFIDF_MODE to "scripted".
TFIDF_MODE = "classic"  # "classic" or "scripted"

# Dirichlet LM parameter. If Canvas specifies a mu, use that value instead.
DIRICHLET_MU = 2000


## 2) Connect to ElasticSearch
Make sure ES is running before executing this cell.

In [ ]:
from elasticsearch import Elasticsearch
from elasticsearch.helpers import streaming_bulk

es = Elasticsearch([ES_HOST])
info = es.info()
print("Connected to cluster:", info.get("cluster_name"))
print("ES version:", info.get("version", {}).get("number"))


## 3) Create 9 indices (3 languages × 3 similarity functions)
Each index stores documents with fields: `qid`, `title`, `body`, `answer`.
We define an analyzer `my_analyzer` (lowercase + porter stem).

In [ ]:
def build_index_body(model: str) -> Dict:
    analysis = {
        "analyzer": {
            "my_analyzer": {
                "type": "custom",
                "tokenizer": "standard",
                "filter": ["lowercase", "porter_stem"],
            }
        }
    }

    similarities = {}

    if model == "bm25":
        similarities["my_similarity"] = {"type": "BM25"}

    elif model == "dirichlet":
        similarities["my_similarity"] = {"type": "LMDirichlet", "mu": DIRICHLET_MU}

    elif model == "tfidf":
        if TFIDF_MODE == "classic":
            similarities["my_similarity"] = {"type": "classic"}
        else:
            # Scripted similarity (TF-IDF-like). This is a reasonable TF-IDF sketch, but
            # if your class expects a specific formula, replace the script accordingly.
            similarities["my_similarity"] = {
                "type": "scripted",
                "script": {
                    "source": """
                        double tf = doc.freq;
                        double norm = 1.0 / Math.sqrt(doc.length);
                        double idf = Math.log((field.docCount + 1.0) / (term.docFreq + 1.0)) + 1.0;
                        return query.boost * tf * norm * idf;
                    """
                }
            }
    else:
        raise ValueError(f"Unknown model: {model}")

    settings = {
        "number_of_shards": 1,
        "analysis": analysis,
        "similarity": similarities,
    }

    mappings = {
        "properties": {
            "qid": {"type": "keyword"},
            "title": {"type": "text", "analyzer": "my_analyzer", "similarity": "my_similarity"},
            "body": {"type": "text", "analyzer": "my_analyzer", "similarity": "my_similarity"},
            "answer": {"type": "text", "analyzer": "my_analyzer", "similarity": "my_similarity"},
        }
    }

    return {"settings": settings, "mappings": mappings}


def recreate_index(idx: str, model: str):
    if es.indices.exists(index=idx):
        es.indices.delete(index=idx)
    es.indices.create(index=idx, body=build_index_body(model))
    print("Created:", idx)


# Create all 9 indices
for lang in LANGS:
    for model in ["tfidf", "bm25", "dirichlet"]:
        recreate_index(index_name(lang, model), model)


## 4) Inspect qid2all format
The handout doesn’t fully specify the delimiter. Preview the first lines.
Then choose a parser.

Two parsers are provided:
- `tab` expects: `qid<TAB>title<TAB>body<TAB>answer`
- `jsonl` expects each line is JSON

In [ ]:
def preview_file(path: Path, n: int = 3):
    print("=== Preview:", path, "===")
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for _ in range(n):
            line = f.readline()
            if not line:
                break
            print(line[:300].rstrip())
    print()

for lang in LANGS:
    preview_file(QID2ALL[lang], n=2)


In [ ]:
def parse_tab_separated(path: Path) -> Iterable[Dict]:
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) < 4:
                continue
            qid, title, body = parts[0], parts[1], parts[2]
            answer = "\t".join(parts[3:])
            yield {"qid": qid, "title": title, "body": body, "answer": answer}

def parse_json_lines(path: Path) -> Iterable[Dict]:
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            qid = str(obj.get("qid") or obj.get("id") or obj.get("_id"))
            yield {
                "qid": qid,
                "title": obj.get("title", ""),
                "body": obj.get("body", ""),
                "answer": obj.get("answer", ""),
            }

PARSER = "tab"  # change to "jsonl" if needed

def iter_docs_for_lang(lang: str) -> Iterable[Dict]:
    p = QID2ALL[lang]
    if PARSER == "tab":
        return parse_tab_separated(p)
    if PARSER == "jsonl":
        return parse_json_lines(p)
    raise ValueError("Unknown PARSER")


## 5) Bulk index all documents
This writes the same docs into the 3 indices for that language (tfidf/bm25/dirichlet).

Start with `max_docs=5000` for a fast validation, then set `max_docs=None` for full indexing.

In [ ]:
def bulk_actions(target_index: str, docs: List[Dict]) -> Iterable[Dict]:
    for d in docs:
        yield {
            "_op_type": "index",
            "_index": target_index,
            "_id": d["qid"],      # critical: qid becomes ES _id so es.get(id=qid) works later
            "_source": d,
        }

def index_language(lang: str, chunk_size: int = 1000, max_docs: Optional[int] = 5000):
    models = ["tfidf", "bm25", "dirichlet"]
    indices = [index_name(lang, m) for m in models]

    buf: List[Dict] = []
    processed = 0

    for d in iter_docs_for_lang(lang):
        buf.append(d)
        if len(buf) >= chunk_size:
            for idx in indices:
                ok = 0
                for success, _ in streaming_bulk(es, bulk_actions(idx, buf), chunk_size=chunk_size, raise_on_error=False):
                    ok += int(success)
                print(f"{lang}: chunk -> {idx}: {ok}/{len(buf)}")
            processed += len(buf)
            buf = []
            if max_docs and processed >= max_docs:
                break

    if buf:
        for idx in indices:
            ok = 0
            for success, _ in streaming_bulk(es, bulk_actions(idx, buf), chunk_size=chunk_size, raise_on_error=False):
                ok += int(success)
            print(f"{lang}: final -> {idx}: {ok}/{len(buf)}")
        processed += len(buf)

    for idx in indices:
        es.indices.refresh(index=idx)

    print(f"Done indexing {lang}. Docs processed: {processed}")

# ---- Run indexing ----
for lang in LANGS:
    index_language(lang, chunk_size=1000, max_docs=5000)  # set max_docs=None for full run


## 6) Parse cosidf judgments into a dict
We assume each line contains at least: `qid1 qid2 label` (whitespace-separated).
For each qid1 you typically have 30 qid2s with labels.

In [ ]:
def parse_cosidf(path: Path) -> Dict[str, List[Tuple[str, int]]]:
    out: Dict[str, List[Tuple[str, int]]] = {}
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = re.split(r"\s+", line)
            if len(parts) < 3:
                continue
            qid1, qid2, label = parts[0], parts[1], parts[2]
            try:
                lbl = int(float(label))
            except ValueError:
                continue
            out.setdefault(qid1, []).append((qid2, lbl))
    return out

cosidf_map = {lang: parse_cosidf(COSIDF[lang]) for lang in LANGS}
for lang in LANGS:
    sample_qid1 = next(iter(cosidf_map[lang]))
    print(lang, "sample qid1:", sample_qid1, "judged docs:", len(cosidf_map[lang][sample_qid1]))


## 7) Rank Eval request body (NDCG@10)
Matches the structure from the handout’s Figure 2:
- exclude the query doc itself
- search `title`, `body`, `answer` with boosts
- metric: DCG@10 normalized (NDCG@10)

In [ ]:
def make_ratings(index: str, judged: List[Tuple[str, int]]) -> List[Dict]:
    return [{"_index": index, "_id": qid2, "rating": int(lbl)} for qid2, lbl in judged]

def ranking_request(qid1: str, qid1_title: str, ratings: List[Dict]) -> Dict:
    return {
        "requests": [
            {
                "id": str(qid1),
                "request": {
                    "query": {
                        "bool": {
                            "must_not": {"match": {"_id": qid1}},
                            "should": [
                                {"match": {"title":  {"query": qid1_title, "boost": 3.0, "analyzer": "my_analyzer"}}},
                                {"match": {"body":   {"query": qid1_title, "boost": 0.5, "analyzer": "my_analyzer"}}},
                                {"match": {"answer": {"query": qid1_title, "boost": 0.5, "analyzer": "my_analyzer"}}},
                            ],
                        }
                    }
                },
                "ratings": ratings,
            }
        ],
        "metric": {"dcg": {"k": 10, "normalize": True}},
    }


## 8) Evaluate mean NDCG@10 for each of the 9 indices
Algorithm: for each qid1, fetch its title from ES, call rank_eval, collect `metric_score`, then average.

In [ ]:
def get_title(idx: str, qid: str) -> Optional[str]:
    try:
        doc = es.get(index=idx, id=qid)
        return doc["_source"].get("title", "")
    except Exception:
        return None

def evaluate_index(lang: str, model: str, max_qids: Optional[int] = 200) -> Dict:
    idx = index_name(lang, model)
    qid1s = list(cosidf_map[lang].keys())
    if max_qids:
        qid1s = qid1s[:max_qids]

    scores: List[float] = []
    missing_titles = 0

    for qid1 in qid1s:
        title = get_title(idx, qid1)
        if not title:
            missing_titles += 1
            continue

        judged = cosidf_map[lang][qid1]
        ratings = make_ratings(idx, judged)
        body = ranking_request(qid1, title, ratings)

        try:
            res = es.rank_eval(index=idx, body=body)
            scores.append(float(res.get("metric_score", 0.0)))
        except Exception:
            scores.append(0.0)

    mean_ndcg = sum(scores) / max(1, len(scores))
    return {
        "index": idx,
        "lang": lang,
        "model": model,
        "mean_ndcg@10": mean_ndcg,
        "num_qids_requested": len(qid1s),
        "num_scored": len(scores),
        "missing_titles": missing_titles,
    }

results = []
for lang in LANGS:
    for model in ["tfidf", "bm25", "dirichlet"]:
        r = evaluate_index(lang, model, max_qids=200)  # set None for full evaluation
        results.append(r)
        print(f"{r['index']}: mean NDCG@10={r['mean_ndcg@10']:.4f} | scored={r['num_scored']} | missing_titles={r['missing_titles']}")


## 9) Print the 9 required scores + write a tiny report file
Submission wants 9 NDCG@10 values (3 langs × 3 similarities).

In [ ]:
# Build a lookup table
score_map = {(r["lang"], r["model"]): r["mean_ndcg@10"] for r in results}

for lang in LANGS:
    print(f"\n== {lang.upper()} ==")
    for model in ["tfidf", "bm25", "dirichlet"]:
        print(f"{model:10s}: {score_map[(lang, model)]:.6f}")

# Save a report text file
out_path = Path("hw2_ndcg_report.txt")
with out_path.open("w", encoding="utf-8") as f:
    for lang in LANGS:
        for model in ["tfidf", "bm25", "dirichlet"]:
            f.write(f"{index_name(lang, model)}\t{score_map[(lang, model)]:.6f}\n")

print("\nSaved report to:", out_path.resolve())


## 10) Troubleshooting
- If ES won’t connect: confirm `ES_HOST` and that ES is running.
- If `missing_titles` is high: your indexing `_id` doesn’t match qids used in cosidf.
- If NDCG is ~0 for most queries: check analyzer name, boosts, ratings index name, and whether docs are actually indexed.
- Once the pipeline works on a subset, rerun with full data (`max_docs=None`, `max_qids=None`).